# 01 — Data cleaning
Run the cells in order from the project root. The supplied verification CSV already contains 1,715 single-disease rows, so this notebook cannot independently recount the original 3,200 multilabel images. It still verifies Positive_Label_Count == 1 and keeps the original train, val and test membership.


## 1. Load metadata
Locate the supplied CSVs. The 1,715-row verification file is the pre-cleaning input. The prior audit is only a cache for valid SHA-256 results.


In [ ]:
from pathlib import Path
import hashlib
import re
import pandas as pd
from PIL import Image

PROJECT = Path.cwd()
ROOT = PROJECT / "RFMiD_share"
RAW_CSV = ROOT / "DATASET" / "RFMiD_21_single_disease_verification.csv"
AUDIT_CSV = ROOT / "DATASET" / "dataset_audit.csv"
FINAL_CSV = ROOT / "DATASET" / "cleaned_top11_metadata.csv"
assert RAW_CSV.is_file(), f"Missing {RAW_CSV}. Start Jupyter in the project root."
raw = pd.read_csv(RAW_CSV, dtype={"Image_ID": "string", "Class_Index": "string"})
print("Input rows:", len(raw))
print("Columns:", raw.columns.tolist())
display(raw.head(3))


## 2. Keep single-label records
Only rows marked with exactly one positive label enter this multiclass dataset. Show the counts before touching image files.


In [2]:
print("Positive label counts:")
display(raw["Positive_Label_Count"].value_counts(dropna=False).sort_index())
single = raw.loc[pd.to_numeric(raw["Positive_Label_Count"], errors="coerce").eq(1)].copy()
print("Original dataset size stated in project brief: 3200 (not reconstructable from this CSV)")
print("Rows in supplied verification CSV:", len(raw))
print("Single-label rows:", len(single))
print("Rows excluded:", len(raw) - len(single))


Positive label counts:


Positive_Label_Count
1    1715
Name: count, dtype: int64

Original dataset size stated in project brief: 3200 (not reconstructable from this CSV)
Rows in supplied verification CSV: 1715
Single-label rows: 1715
Rows excluded: 0


## 3. Check actual images
Try the supplied path columns in the required order, then the existing cleaned folder as a fallback. PIL reads the image data so a file with an image extension alone does not pass.


In [ ]:
audit = pd.read_csv(AUDIT_CSV, dtype={"Image_ID": "string", "Class_Index": "string"}) if AUDIT_CSV.is_file() else pd.DataFrame()
def resolve_image_path(row):
    candidates = []
    for column in ["Image_Path", "Output_Path", "Resolved_Source_Path", "Source_Path"]:
        value = row.get(column)
        if pd.notna(value) and str(value).strip():
            candidates.append(str(value))
    for value in candidates:
        path = Path(value)
        choices = [path] if path.is_absolute() else [ROOT / path, PROJECT / path]
        for choice in choices:
            if choice.is_file():
                return choice
    return None

paths, sizes, missing = [], [], []
for _, row in single.iterrows():
    path = resolve_image_path(row)
    try:
        if path is None:
            raise FileNotFoundError("No candidate path exists")
        with Image.open(path) as image:
            image.load()
            sizes.append(image.size)
        paths.append(path)
        missing.append("")
    except (OSError, ValueError, FileNotFoundError) as error:
        paths.append(None)
        sizes.append((None, None))
        missing.append(str(error))
single["Image_Path"] = [p.relative_to(ROOT).as_posix() if p is not None and p.is_relative_to(ROOT) else (str(p) if p is not None else "") for p in paths]
single["Width"] = [size[0] for size in sizes]
single["Height"] = [size[1] for size in sizes]
single["Image_Error"] = missing
unreadable = single.loc[single["Image_Error"].ne("")]
readable = single.loc[single["Image_Error"].eq("")].copy()
print("Before image check:", len(single))
print("Readable:", len(readable))
print("Missing or unreadable:", len(unreadable))
display(unreadable[["Split", "Image_ID", "Disease", "Image_Error"]].head(20))
print("Remaining:", len(readable))


## 4. Validate metadata
Show each requested check directly. Disease and Class_Index should form a one-to-one mapping; duplicate IDs are checked within each original split.


In [4]:
required = ["Split", "Image_ID", "Filename", "Disease", "Class_Index", "Image_Path"]
print("Nulls by column:")
display(readable[required].isna().sum().to_frame("null_count"))
print("Empty strings by text column:")
display(readable[required].astype("string").apply(lambda col: col.str.strip().eq("").sum()).to_frame("empty_count"))
invalid_split = readable.loc[~readable["Split"].isin(["train", "val", "test"])]
class_num = pd.to_numeric(readable["Class_Index"], errors="coerce")
invalid_class = readable.loc[class_num.isna() | (class_num < 0) | (class_num % 1 != 0)]
invalid_id = readable.loc[~readable["Image_ID"].astype("string").str.fullmatch(r"[1-9][0-9]*", na=False)]
duplicate_ids = readable.loc[readable.duplicated(["Split", "Image_ID"], keep=False)]
mapping = readable.groupby(["Disease", "Class_Index"]).size().reset_index(name="Count")
bad_diseases = mapping.groupby("Disease")["Class_Index"].nunique().loc[lambda x: x > 1]
bad_classes = mapping.groupby("Class_Index")["Disease"].nunique().loc[lambda x: x > 1]
print("Invalid splits:", len(invalid_split), "| invalid Class_Index:", len(invalid_class), "| invalid Image_ID:", len(invalid_id))
print("Duplicate IDs within splits:", len(duplicate_ids))
print("Diseases with multiple class indexes:", len(bad_diseases), "| class indexes with multiple diseases:", len(bad_classes))
display(mapping.sort_values("Class_Index", key=lambda x: pd.to_numeric(x)))
assert readable[required].notna().all().all()
assert not readable[required].astype("string").apply(lambda col: col.str.strip().eq("").any()).any()
assert len(invalid_split) == len(invalid_class) == len(invalid_id) == len(duplicate_ids) == len(bad_diseases) == len(bad_classes) == 0


Nulls by column:


,null_count
Split,0
Image_ID,0
Filename,0
Disease,0
Class_Index,0
Image_Path,0


Empty strings by text column:


,empty_count
Split,0
Image_ID,0
Filename,0
Disease,0
Class_Index,0
Image_Path,0


Invalid splits: 0 | invalid Class_Index: 0 | invalid Image_ID: 0
Duplicate IDs within splits: 0
Diseases with multiple class indexes: 0 | class indexes with multiple diseases: 0


,Disease,Class_Index,Count
8,DR,0,401
2,ARMD,1,69
11,MH,2,315
7,DN,3,152
13,MYA,4,75
3,BRVO,5,82
20,TSLN,6,40
9,ERM,7,15
10,LS,8,12
12,MS,9,12


## 5. SHA-256 duplicates and leakage
Use cached hashes from the supplied audit only for matching split, ID, disease and known image path. Otherwise hash the current file bytes. Set FORCE_REHASH to True if image files have changed since the audit.


In [5]:
FORCE_REHASH = False
cached_hashes = {}
if not audit.empty and not FORCE_REHASH and "Image_Hash" in audit:
    for _, row in audit.iterrows():
        value = str(row["Image_Hash"]).lower()
        if re.fullmatch(r"[0-9a-f]{64}", value):
            key = (row["Split"], row["Image_ID"], row["Disease"])
            cached_hashes[key] = (value, {str(row.get("Source_Path", "")), str(row.get("Output_Path", ""))})

hashes, reused = [], 0
for _, row in readable.iterrows():
    key = (row["Split"], row["Image_ID"], row["Disease"])
    cached = cached_hashes.get(key)
    path_string = row["Image_Path"]
    if cached and path_string in cached[1]:
        digest = cached[0]
        reused += 1
    else:
        digest_obj = hashlib.sha256()
        with open(ROOT / path_string, "rb") as file:
            for chunk in iter(lambda: file.read(1024 * 1024), b""):
                digest_obj.update(chunk)
        digest = digest_obj.hexdigest()
    hashes.append(digest)
readable["Image_Hash"] = hashes
print("Reused prior valid hashes:", reused, "| hashed file bytes now:", len(readable) - reused)
print("All hashes valid:", readable["Image_Hash"].str.fullmatch(r"[0-9a-f]{64}").all())
assert readable["Image_Hash"].str.fullmatch(r"[0-9a-f]{64}").all()


Reused prior valid hashes: 1700 | hashed file bytes now: 0
All hashes valid: True


In [6]:
print("Duplicates within each split (extra images):")
within_count = 0
for split in ["train", "val", "test"]:
    part = readable.loc[readable["Split"].eq(split)]
    count = int(part.duplicated("Image_Hash").sum())
    within_count += count
    print(split, count)
print("Cross-split shared hashes:")
split_hashes = {split: set(readable.loc[readable["Split"].eq(split), "Image_Hash"]) for split in ["train", "val", "test"]}
cross_counts = {}
for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
    cross_counts[(left, right)] = len(split_hashes[left] & split_hashes[right])
    print(left, "vs", right, cross_counts[(left, right)])
leakage_count = len(set.union(*(split_hashes[a] & split_hashes[b] for a, b in cross_counts)))
print("Total distinct leaking hashes:", leakage_count)
assert within_count == 0 and leakage_count == 0, "Review duplicate rows before selecting classes"


Duplicates within each split (extra images):
train 0
val 0
test 0
Cross-split shared hashes:
train vs val 0
train vs test 0
val vs test 0
Total distinct leaking hashes: 0


## 6. Choose the 11 largest classes
Count valid single-label images across all existing splits. Ties are resolved by disease name so target numbers stay stable.


In [7]:
class_counts = readable["Disease"].value_counts().rename_axis("Disease").reset_index(name="Count")
class_counts = class_counts.sort_values(["Count", "Disease"], ascending=[False, True]).reset_index(drop=True)
display(class_counts)
top11 = class_counts.head(11)["Disease"].tolist()
TARGET_MAP = {disease: index for index, disease in enumerate(top11)}
final = readable.loc[readable["Disease"].isin(top11)].copy()
final["Target"] = final["Disease"].map(TARGET_MAP).astype("int64")
print("TARGET_MAP:", TARGET_MAP)
assert len(TARGET_MAP) == 11
assert set(final["Target"]) == set(range(11))
assert final.groupby("Disease")["Target"].nunique().eq(1).all()
assert final.groupby("Target")["Disease"].nunique().eq(1).all()
display(final.groupby(["Target", "Disease", "Class_Index"]).size().reset_index(name="Count"))


,Disease,Count
0,DR,401
1,MH,315
2,ODC,155
3,DN,152
4,BRVO,82
5,ODE,81
6,MYA,75
7,ARMD,69
8,RS,67
9,CSR,54


TARGET_MAP: {'DR': 0, 'MH': 1, 'ODC': 2, 'DN': 3, 'BRVO': 4, 'ODE': 5, 'MYA': 6, 'ARMD': 7, 'RS': 8, 'CSR': 9, 'CRVO': 10}


,Target,Disease,Class_Index,Count
0,0,DR,0,401
1,1,MH,2,315
2,2,ODC,11,155
3,3,DN,3,152
4,4,BRVO,5,82
5,5,ODE,16,81
6,6,MYA,4,75
7,7,ARMD,1,69
8,8,RS,13,67
9,9,CSR,10,54


## 7. Keep original splits and inspect sizes
The expected counts are checks, never values forced onto the data. Original image dimensions remain in the metadata; no images are resized or overwritten here.


In [8]:
counts = final["Split"].value_counts().reindex(["train", "val", "test"], fill_value=0)
expected = pd.Series({"train": 911, "val": 278, "test": 304})
display(pd.DataFrame({"actual": counts, "previously_expected": expected, "difference": counts - expected}))
print("Final total:", len(final), "| previously expected: 1493")
print("Original resolutions, most common first:")
display(final.groupby(["Width", "Height"]).size().sort_values(ascending=False).rename("Count").reset_index().head(20))


,actual,previously_expected,difference
train,911,911,0
val,278,278,0
test,304,304,0


Final total: 1493 | previously expected: 1493
Original resolutions, most common first:


,Width,Height,Count
0,2144.0,1424.0,1225
1,4288.0,2848.0,196
2,2048.0,1536.0,72


## 8. Save final metadata
Save relative image paths, original class indexes, remapped targets and hashes in one CSV. The next notebook reads this file.


In [9]:
final = final.drop(columns=["Image_Error"])
final.to_csv(FINAL_CSV, index=False)
check = pd.read_csv(FINAL_CSV)
print("Saved:", FINAL_CSV)
print("Rows reloaded:", len(check), "| columns:", check.columns.tolist())
assert len(check) == len(final)
print(f"Original dataset (project brief): 3200 | supplied CSV: {len(raw)} | single-label: {len(single)} | readable: {len(readable)} | missing/unreadable removed: {len(unreadable)}")
print(f"Within-split duplicate images: {within_count} | cross-split leakage hashes: {leakage_count}")
print(f"Selected classes: {len(TARGET_MAP)} | train: {counts['train']} | val: {counts['val']} | test: {counts['test']} | total: {len(final)}")


Saved: c:\Users\aravi\OneDrive\Desktop\coding\FYP MR disease classification\cleaned_top11_metadata.csv
Rows reloaded: 1493 | columns: ['Split', 'Image_ID', 'Filename', 'Disease', 'Class_Index', 'Positive_Label_Count', 'Source_Path', 'Output_Path', 'Image_Path', 'Width', 'Height', 'Image_Hash', 'Target']
Original dataset (project brief): 3200 | supplied CSV: 1715 | single-label: 1715 | readable: 1700 | missing/unreadable removed: 15
Within-split duplicate images: 0 | cross-split leakage hashes: 0
Selected classes: 11 | train: 911 | val: 278 | test: 304 | total: 1493
